# 数据准备：PEAD Baseline 回归表

**目标**：把 WRDS 原始数据整理成一张**事件级大表**（一行 = 一个盈余公告事件），
供 `回归准备.ipynb` 直接选列跑回归。

**当前范围**：baseline 回归所需的 CAR / SUE / Controls。
Moderator（ATT / ADOPT / OUT）与 LLM signal 后续以加列方式补充，不需重造此表。

| 步骤 | 内容 | 产出 |
|---|---|---|
| 1 | 股票池 + 跨库连接表 | `build/universe.parquet`, `build/link_*.parquet` |
| 2 | BE / ME / BM → 25 组 size×B/M → 两套基准组合日收益 | `build/port25_*.parquet` |
| 3 | 事件表（IBES actuals → PERMNO） | `build/events.parquet` |
| 4 | CAR：buy-and-hold [0,1] 与 [2,61]，C2C 与 O2O 各一套 | `build/car.parquet` |
| 5 | SUE：60 天窗口共识 → (e−F)/P_adj → 季度十分位 | `build/sue.parquet` |
| 6 | 10 个控制变量 | `build/ctrl_*.parquet` |
| 7 | 合并 + 诊断 | `build/pead_panel.parquet` |

**收益数据**：C2C 用 `data/crsp_daily_*.parquet` 的 `dlyret`；
O2O 用 `export/crsp_daily_ret_c2c_o2o.parquet` 的 `O2O_RET`（来自 yifei v5，1996–2026-03）。


## 0. Setup

In [1]:
# scipy 必须在 pandas/pyarrow 之前 import：pyarrow 会先加载系统旧版 libstdc++，
# 之后再 import scipy 会报 GLIBCXX_3.4.29 not found
import scipy.stats  # noqa: F401

import gc
import os
import glob

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA   = "data"      # WRDS 原始下载
BUILD  = "build"     # 中间产物
EXPORT = "export"    # C2C + O2O 日收益面板
os.makedirs(BUILD, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

print("原始数据文件:", len([f for f in os.listdir(DATA) if f.endswith(".parquet")]), "个")

原始数据文件: 102 个


## 1. 股票池 + 跨库连接表

### 1.1 普通股口径

CRSP CIZ 格式没有旧版的 `shrcd`，普通股要靠几个字段联合判断。各层筛选后的 permno 数：

| 条件 | permno 数 |
|---|---|
| 全部 | 40,518 |
| `sharetype='NS'` & `securitytype='EQTY'` & `securitysubtype='COM'` | 30,263 |
| + `usincflg='Y'`（美国注册） | 27,370 |
| + `issuertype ∈ {CORP, ACOR}`（排除 REIT） | **26,914** |

最后一档对应旧版 SIZ 的 `shrcd ∈ {10, 11}`，也是 HLT 2009 的样本口径。

**注意**：这里不做硬删除，而是把 `is_common` / `is_us` / `is_operating` / `in_universe` /
`is_nyse` / `on_major_exch` 都存成布尔列。想放宽口径（比如把 REIT 纳回来）时改筛选条件即可，
不必重跑。`is_nyse` 单独留着是因为 25 组的 breakpoint 只能用 NYSE 股票算。

In [2]:
s = pd.read_parquet(f"{DATA}/crsp_security_info.parquet")
s["secinfostartdt"] = pd.to_datetime(s["secinfostartdt"])
s["secinfoenddt"]   = pd.to_datetime(s["secinfoenddt"])

s["is_common"]      = (s["sharetype"] == "NS") & (s["securitytype"] == "EQTY") & (s["securitysubtype"] == "COM")
s["is_us"]          = s["usincflg"] == "Y"
s["is_operating"]   = s["issuertype"].isin(["CORP", "ACOR"])      # 排除 REIT
s["in_universe"]    = s["is_common"] & s["is_us"] & s["is_operating"]
s["is_nyse"]        = s["primaryexch"] == "N"                     # breakpoint universe
s["on_major_exch"]  = s["primaryexch"].isin(["N", "A", "Q"])      # NYSE / AMEX / NASDAQ

keep = ["permno", "permco", "cusip", "siccd", "primaryexch",
        "secinfostartdt", "secinfoenddt",
        "sharetype", "securitytype", "securitysubtype", "usincflg", "issuertype",
        "is_common", "is_us", "is_operating", "in_universe", "is_nyse", "on_major_exch"]
universe = s[keep].sort_values(["permno", "secinfostartdt"]).reset_index(drop=True)
universe.to_parquet(f"{BUILD}/universe.parquet", index=False)

print(f"universe.parquet: {len(universe):,} 条 security-info 区间, {universe['permno'].nunique():,} 个 permno")
print(f"  最终宇宙 in_universe=True: {universe.loc[universe['in_universe'], 'permno'].nunique():,} 个 permno")
print(f"  其中 NYSE (breakpoint 用): {universe.loc[universe['in_universe'] & universe['is_nyse'], 'permno'].nunique():,} 个")
print("\n最终宇宙按主交易所:")
print(universe[universe["in_universe"]].groupby("primaryexch")["permno"].nunique().sort_values(ascending=False).to_string())

universe.parquet: 193,968 条 security-info 区间, 41,520 个 permno
  最终宇宙 in_universe=True: 27,016 个 permno
  其中 NYSE (breakpoint 用): 6,544 个

最终宇宙按主交易所:
primaryexch
Q    19975
N     6544
A     4186
X     1232
R       18
B        4
I        1


### 1.2 连接表

两处需要处理：

- **`linkenddt` / `edate` 为空** 表示链接至今仍有效。留成 `NaT` 会让后续按日期区间筛选时
  整行被丢掉，所以统一填成 `2099-12-31`。
- **一对多**：同一 permno 可能在不同时期对应不同 gvkey（并购、重组），同一 IBES ticker
  也可能对应多个 permno。这些是真实的历史变更，不能去重——后续 merge 必须**按日期区间**匹配，
  而不是简单 `merge(on='permno')`。

In [3]:
# CRSP <-> Compustat
ccm = pd.read_parquet(f"{DATA}/link_crsp_compustat.parquet")
ccm["linkdt"]    = pd.to_datetime(ccm["linkdt"])
ccm["linkenddt"] = pd.to_datetime(ccm["linkenddt"]).fillna(pd.Timestamp("2099-12-31"))
ccm["permno"]    = ccm["permno"].astype("int64")
ccm = ccm.sort_values(["permno", "linkdt"]).reset_index(drop=True)
ccm.to_parquet(f"{BUILD}/link_ccm.parquet", index=False)

# IBES <-> CRSP
ibl = pd.read_parquet(f"{DATA}/link_ibes_crsp.parquet")
ibl["sdate"]  = pd.to_datetime(ibl["sdate"])
ibl["edate"]  = pd.to_datetime(ibl["edate"]).fillna(pd.Timestamp("2099-12-31"))
ibl["permno"] = ibl["permno"].astype("int64")
ibl = ibl.sort_values(["ticker", "sdate"]).reset_index(drop=True)
ibl.to_parquet(f"{BUILD}/link_ibes.parquet", index=False)

print(f"link_ccm : {len(ccm):,} 行 | gvkey {ccm['gvkey'].nunique():,} | permno {ccm['permno'].nunique():,}")
print(f"  仍有效的链接 (linkenddt 原为空): {(ccm['linkenddt'] == pd.Timestamp('2099-12-31')).sum():,}")
print(f"  一个 permno 对多个 gvkey: {(ccm.groupby('permno')['gvkey'].nunique() > 1).sum():,} 个 permno")
print(f"\nlink_ibes: {len(ibl):,} 行 | ticker {ibl['ticker'].nunique():,} | permno {ibl['permno'].nunique():,}")
print(f"  一个 ticker 对多个 permno: {(ibl.groupby('ticker')['permno'].nunique() > 1).sum():,} 个 ticker")

link_ccm : 33,324 行 | gvkey 29,163 | permno 29,567
  仍有效的链接 (linkenddt 原为空): 6,383
  一个 permno 对多个 gvkey: 745 个 permno

link_ibes: 30,080 行 | ticker 21,078 | permno 21,555
  一个 ticker 对多个 permno: 631 个 ticker


### 1.3 按日期区间贴 PERMNO 的工具函数

后续第 3 步（IBES 事件 → permno）和第 6 步（Compustat → permno、13F cusip → permno）都要用。
逻辑：先按 key 做多对多 merge，再用日期落在 `[start, end]` 区间内这一条件筛出唯一匹配。

In [4]:
def attach_by_date(df, link, key, date_col, start_col, end_col, keep_cols=None):
    """按日期区间把 link 表的列贴到 df 上。

    返回值行数可能少于输入（无匹配的行被丢弃），也可能多于输入
    （同一时点存在多条有效链接——需要在调用处检查并处理）。
    """
    m = df.merge(link, on=key, how="inner", suffixes=("", "_lnk"))
    m = m[(m[date_col] >= m[start_col]) & (m[date_col] <= m[end_col])]
    drop = [start_col, end_col] + ([] if keep_cols is None else
                                   [c for c in link.columns if c not in keep_cols + [key, start_col, end_col]])
    return m.drop(columns=[c for c in drop if c in m.columns])


# 自检：IBES 事件贴 permno 的匹配率与唯一性
_act = pd.read_parquet(f"{DATA}/ibes_actuals.parquet", columns=["ticker", "anndats", "pends"])
_act["anndats"] = pd.to_datetime(_act["anndats"])
_ev = attach_by_date(_act, ibl, key="ticker", date_col="anndats", start_col="sdate", end_col="edate")

n_in  = len(_act)
n_out = len(_ev)
n_uni = _ev.groupby(["ticker", "anndats"]).ngroups
print(f"IBES 事件 {n_in:,} 条 → 贴上 permno {n_out:,} 条 (唯一事件 {n_uni:,}, 匹配率 {n_uni/n_in:.1%})")
print(f"  同一事件匹配到多个 permno 的比例: {(n_out - n_uni)/n_out:.4%}")

IBES 事件 742,513 条 → 贴上 permno 619,986 条 (唯一事件 605,899, 匹配率 81.6%)
  同一事件匹配到多个 permno 的比例: 2.2721%


## 2. BE / ME / BM → 25 组 size×B/M → 基准组合日收益

### Formation 规则（HLT 2009 Sec III / Fama-French 标准）

| 用途 | 时点 |
|---|---|
| Size 排序的 ME | formation 当年 **6 月末**市值 |
| B/M 分子 BE | formation **前一年**财年末的账面权益 |
| B/M 分母 ME | formation **前一年 12 月末**市值 |
| Breakpoint | 只用 NYSE 股票算 20/40/60/80 分位 |
| 生效期 | 当年 **7 月 至 次年 6 月**，成员不变 |

`BE = SEQ + TXDITC − PS`，其中
SEQ 取 `seq` → 缺失用 `ceq+pstk` → 再缺用 `at−lt`；
PS 取 `pstkrv` → `pstkl` → `pstk`；`txditc` 缺失记 0。

**单位**：Compustat 是百万美元，CRSP `mthcap` 是千美元，所以 `bm = be × 1000 / me_dec`。

In [5]:
FORCE = False   # True 则忽略缓存重算

# ---------- 2.1 BE ----------
fa = pd.read_parquet(f"{DATA}/compustat_annual.parquet")
fa["datadate"] = pd.to_datetime(fa["datadate"])

seq = fa["seq"].fillna(fa["ceq"] + fa["pstk"]).fillna(fa["at"] - fa["lt"])
ps  = fa["pstkrv"].fillna(fa["pstkl"]).fillna(fa["pstk"])
fa["be"] = seq + fa["txditc"].fillna(0) - ps.fillna(0)
fa["fy_end_year"] = fa["datadate"].dt.year

be_tab = fa.loc[fa["be"].notna(), ["gvkey", "datadate", "fy_end_year", "be", "sich"]]
# 同一 gvkey 同一日历年多条（财年变更）时保留最晚的 datadate
be_tab = be_tab.sort_values("datadate").drop_duplicates(["gvkey", "fy_end_year"], keep="last")
print(f"BE: {len(be_tab):,} 条 gvkey-年 (BE>0 占 {(be_tab['be'] > 0).mean():.3f})")

# ---------- 2.2 gvkey → permno ----------
m = be_tab.merge(ccm, on="gvkey", how="inner")
m = m[(m["datadate"] >= m["linkdt"]) & (m["datadate"] <= m["linkenddt"])]
m = m.sort_values("linkprim").drop_duplicates(["gvkey", "fy_end_year"], keep="first")  # P 优先于 C
be_p = m[["permno", "fy_end_year", "be", "datadate", "gvkey"]]
print(f"BE 贴 permno: {len(be_p):,} 条 ({be_p['permno'].nunique():,} 个 permno)")

# ---------- 2.3 ME: 6月末 / 12月末 ----------
mth = pd.read_parquet(f"{DATA}/crsp_monthly.parquet",
                      columns=["permno", "mthcaldt", "mthcap", "primaryexch"])
mth["mthcaldt"] = pd.to_datetime(mth["mthcaldt"])
me_jun = (mth[mth["mthcaldt"].dt.month == 6]
          .assign(ffyear=lambda x: x["mthcaldt"].dt.year)[["permno", "ffyear", "mthcap", "primaryexch"]]
          .rename(columns={"mthcap": "me_june"}))
me_dec = (mth[mth["mthcaldt"].dt.month == 12]
          .assign(fy_end_year=lambda x: x["mthcaldt"].dt.year)[["permno", "fy_end_year", "mthcap"]]
          .rename(columns={"mthcap": "me_dec"}))

# ---------- 2.4 BM ----------
bm = be_p.merge(me_dec, on=["permno", "fy_end_year"], how="inner")
bm["ffyear"] = bm["fy_end_year"] + 1
bm["bm_raw"] = bm["be"] * 1000.0 / bm["me_dec"]        # 百万 → 千
bm = bm[(bm["be"] > 0) & (bm["me_dec"] > 0)]
print(f"BM: {len(bm):,} 条 permno-formation年, 中位数 {bm['bm_raw'].median():.3f}")

BE: 308,363 条 gvkey-年 (BE>0 占 0.872)


BE 贴 permno: 193,829 条 (20,428 个 permno)


BM: 175,254 条 permno-formation年, 中位数 0.596


In [6]:
# ---------- 2.5 25 组 formation ----------
uni_pm = universe.loc[universe["in_universe"], ["permno", "secinfostartdt", "secinfoenddt", "is_nyse"]]

f = me_jun.merge(bm[["permno", "ffyear", "bm_raw", "be", "me_dec"]], on=["permno", "ffyear"], how="inner")
f = f[f["me_june"] > 0].copy()
f["jun_date"] = pd.to_datetime(f["ffyear"].astype(str) + "-06-30")
f = f.merge(uni_pm, on="permno", how="inner")
f = f[(f["jun_date"] >= f["secinfostartdt"]) & (f["jun_date"] <= f["secinfoenddt"])]
f = f.drop_duplicates(["permno", "ffyear"])


def assign_quintile(g, col):
    """用 NYSE 股票算 20/40/60/80 分位点，再给全市场打标签。"""
    nyse = g.loc[g["is_nyse"], col]
    if len(nyse) < 25:
        return pd.Series(np.nan, index=g.index)
    bps = np.percentile(nyse, [20, 40, 60, 80])
    return pd.Series(np.searchsorted(bps, g[col], side="right") + 1, index=g.index)


f["size_q"] = f.groupby("ffyear", group_keys=False)[["me_june", "is_nyse"]].apply(lambda g: assign_quintile(g, "me_june"))
f["bm_q"]   = f.groupby("ffyear", group_keys=False)[["bm_raw", "is_nyse"]].apply(lambda g: assign_quintile(g, "bm_raw"))
f = f.dropna(subset=["size_q", "bm_q"])
f["port25"] = ((f["size_q"] - 1) * 5 + f["bm_q"]).astype(int)

membership = f[["permno", "ffyear", "me_june", "me_dec", "be", "bm_raw",
                "size_q", "bm_q", "port25", "is_nyse"]].copy()
membership.to_parquet(f"{BUILD}/port25_membership.parquet", index=False)

print(f"25 组成员表: {len(membership):,} 条, formation 年 {membership['ffyear'].min()}–{membership['ffyear'].max()}")
print(f"每组平均成员数: {membership.groupby(['ffyear', 'port25']).size().mean():.1f}")
print("\n2015 年各组成员数 (行=size五分位, 列=BM五分位):")
print(membership[membership["ffyear"] == 2015]
      .pivot_table(index="size_q", columns="bm_q", values="permno", aggfunc="count").astype("Int64").to_string())

25 组成员表: 135,230 条, formation 年 1995–2026
每组平均成员数: 169.0

2015 年各组成员数 (行=size五分位, 列=BM五分位):
bm_q      1    2    3    4    5
size_q                         
1       277  211  292  355  549
2       157  117  143  136   87
3       106   92   88   77   56
4       120  104   73   50   36
5       116   86   50   44   33


### 2.6 基准组合日收益（C2C 与 O2O 各一套）

对每个交易日、每个组合取**等权平均**收益（HLT 2009 用等权）。
`ffyear` 的归属规则：7 月至次年 6 月同属一个 formation 年。

两套收益必须**分别**用各自口径算，不能混用：
- C2C → `data/crsp_daily_*.parquet` 的 `dlyret`
- O2O → `export/crsp_daily_ret_c2c_o2o.parquet` 的 `O2O_RET`

In [7]:
BENCH_PATH = f"{BUILD}/port25_bench_returns.parquet"

def ffyear_of(dates):
    y = dates.dt.year
    return np.where(dates.dt.month >= 7, y, y - 1)


if os.path.exists(BENCH_PATH) and not FORCE:
    bench = pd.read_parquet(BENCH_PATH)
    bench["date"] = pd.to_datetime(bench["date"])
    print(f"[缓存] {BENCH_PATH}")
else:
    import glob, pyarrow.parquet as pq
    mem = membership[["permno", "ffyear", "port25"]].assign(permno=lambda x: x["permno"].astype("int64"))

    # C2C：逐年读日频
    rows = []
    for path in sorted(glob.glob(f"{DATA}/crsp_daily_*.parquet")):
        d = pd.read_parquet(path, columns=["permno", "dlycaldt", "dlyret"]).dropna(subset=["dlyret"])
        d["dlycaldt"] = pd.to_datetime(d["dlycaldt"]); d["permno"] = d["permno"].astype("int64")
        d["ffyear"] = ffyear_of(d["dlycaldt"])
        d = d.merge(mem, on=["permno", "ffyear"], how="inner")
        rows.append(d.groupby(["dlycaldt", "port25"])["dlyret"].agg(["mean", "size"]).reset_index())
    c2c = pd.concat(rows, ignore_index=True).rename(
        columns={"dlycaldt": "date", "mean": "bench_ret_c2c", "size": "n_c2c"})

    # O2O：单遍扫 export 面板
    acc = []
    pf = pq.ParquetFile(f"{EXPORT}/crsp_daily_ret_c2c_o2o.parquet")
    for b in pf.iter_batches(batch_size=3_000_000, columns=["PERMNO", "date", "O2O_RET"]):
        d = b.to_pandas().rename(columns={"PERMNO": "permno"}).dropna(subset=["O2O_RET"])
        if d.empty: continue
        d["date"] = pd.to_datetime(d["date"]); d["permno"] = d["permno"].astype("int64")
        d["ffyear"] = ffyear_of(d["date"])
        d = d.merge(mem, on=["permno", "ffyear"], how="inner")
        if d.empty: continue
        acc.append(d.groupby(["date", "port25"])["O2O_RET"].agg(["sum", "size"]).reset_index())
    o2o = pd.concat(acc, ignore_index=True).groupby(["date", "port25"], as_index=False)[["sum", "size"]].sum()
    o2o["bench_ret_o2o"] = o2o["sum"] / o2o["size"]
    o2o = o2o.rename(columns={"size": "n_o2o"})[["date", "port25", "bench_ret_o2o", "n_o2o"]]

    bench = c2c.merge(o2o, on=["date", "port25"], how="outer").sort_values(["date", "port25"])
    bench.to_parquet(BENCH_PATH, index=False)

print(f"基准组合日收益: {len(bench):,} 行, {bench['date'].min().date()} ~ {bench['date'].max().date()}")
print("每日组合数分布 (应恒为 25):", bench.groupby("date")["port25"].nunique().value_counts().to_dict())

gc.collect()

[缓存] build/port25_bench_returns.parquet
基准组合日收益: 191,825 行, 1996-01-02 ~ 2026-06-30
每日组合数分布 (应恒为 25): {25: 7673}


0

### 2.7 诊断

三项检查：

1. **size / value 效应是否出现** —— 若 formation 做对了，小市值组和高 B/M 组的平均收益应更高
2. **C2C 与 O2O 是否在追同一个组合** —— 同日相关会很低（两者只共享隔夜段），
   但 21 日累计相关应该很高
3. **两个口径的长期均值差异** —— 微观结构噪音在最小市值组会放大

In [8]:
sub = bench[(bench["date"] >= "2000-01-01") & (bench["date"] <= "2019-12-31")].copy()

ann = (sub.groupby("port25")[["bench_ret_c2c", "bench_ret_o2o"]].mean() * 252).reset_index()
ann["size_q"] = ((ann["port25"] - 1) // 5) + 1
ann["bm_q"]   = ((ann["port25"] - 1) % 5) + 1

print("年化平均收益 C2C (行=size, 列=BM), 2000–2019:")
print(ann.pivot(index="size_q", columns="bm_q", values="bench_ret_c2c").round(3).to_string())
print("\nO2O − C2C 的年化差 (行=size, 列=BM):")
d = ann.assign(diff=ann["bench_ret_o2o"] - ann["bench_ret_c2c"])
print(d.pivot(index="size_q", columns="bm_q", values="diff").round(3).to_string())

s = sub[sub["port25"] == 13].set_index("date").sort_index()
cum_c = (1 + s["bench_ret_c2c"]).rolling(21).apply(np.prod, raw=True) - 1
cum_o = (1 + s["bench_ret_o2o"]).rolling(21).apply(np.prod, raw=True) - 1
print(f"\n组合13: 同日相关 {s[['bench_ret_c2c','bench_ret_o2o']].corr().iloc[0,1]:.3f}"
      f" | 21日累计相关 {cum_c.corr(cum_o):.3f}"
      f" | corr(O2O_t, C2C_t-1) {s['bench_ret_o2o'].corr(s['bench_ret_c2c'].shift(1)):.3f}")

年化平均收益 C2C (行=size, 列=BM), 2000–2019:
bm_q        1      2      3      4      5
size_q                                   
1       0.125  0.162   0.17  0.192  0.243
2       0.104  0.139  0.153  0.131  0.129
3       0.097  0.134  0.142  0.129   0.16
4       0.121  0.131  0.127  0.128  0.121
5       0.094  0.111  0.115  0.109   0.12

O2O − C2C 的年化差 (行=size, 列=BM):
bm_q        1      2      3      4      5
size_q                                   
1       0.058   0.04   0.03  0.027  0.052
2       0.009  0.001  0.001  0.005  0.006
3       0.004  0.003  0.002  0.001 -0.007
4         0.0  0.002  0.002  0.002  0.008
5         0.0  0.001  0.004  0.003  0.007



组合13: 同日相关 0.233 | 21日累计相关 0.958 | corr(O2O_t, C2C_t-1) 0.771


## 3. 事件表

以 I/B/E/S actuals 的公告日 `anndats` 为锚点，贴上 PERMNO，处理三类重复，再映射到交易日。

**三类重复**

| 类型 | 处理 |
|---|---|
| A. 一个事件匹配多个 PERMNO | 先用最终宇宙筛（普通股/美国/非 REIT），残余按 permno 升序取第一个 |
| B. 同一 permno-财季多次公告（重述、更正） | 保留**最早**一次——PEAD 关心的是信息首次到达 |
| C. 同一 permno 同一天公告多个财季（补报） | **全部保留**。公司确实披露了，滞后由 LAG/LAG²/LAG³ 吸收。两列标记：`flag_same_day_multi`（是否属于这种情形）、`is_latest_pends_on_day`（是否当期财季）。注意这些行共享同一个 CAR，回归时按 permno×日期聚类标准误 |

**交易日映射**：`td0` = 公告日当天或之后的第一个交易日。约 1.6% 的公告落在周末/假日，需要顺延。
`td0_idx` 是它在交易日历中的位置，第 4 步算 CAR 窗口时直接用下标取 [0,1] 和 [2,61]。

**IBES 连接表的时间边界**：`wrdsapps.ibcrsphist` 只更新到 **2025-12-31**（4,386 条链接的
`edate` 停在这天），2026 年的公告因此无法映射到 PERMNO。`EXTEND_IBES_LINK=True`
把这些"末期仍有效"的链接延长到 2099-12-31，代价是极小概率的错配
（2026 年内 ticker 被重新分配给另一家公司）。两个标记列的区别：

- `flag_link_extended`：链接在末期仍有效（约 16.8 万个事件）。绝大多数本来就能匹配，**无风险**
- `flag_beyond_link_end`：公告日晚于 2025-12-31，**只因延长才存在**（5,846 个）。要保守就剔除这些

**关于未匹配的 12 万个事件**：已验证不是连接失败，而是这些证券本就不在 CRSP 里。
用 CUSIP（比 ticker 可靠得多，ticker 会被回收复用）回捞的结果：
122,527 个未匹配事件中只有 6,166 个能定位到宇宙内的 PERMNO，其中 5,971 个在 2026 年
（已由链接延长覆盖）——**1995–2025 三十年只多捞回约 195 个事件**。
未匹配样本的公司名（BESPAK、CLUB-MED、KONINKELIJKE WES）也印证它们是外国公司：
`ibes.act_epsus` 的 "US" 指数据源是美国库，不是公司注册地。
项目里虽有 `TICKER_to_PERMNO_*.parquet` 等映射，但既然 CUSIP 都捞不回来，
用 ticker 只会引入错配，不予采用。

In [9]:
# 要保守就 df[~df.flag_beyond_link_end]，等于样本截到 2025-12-31

In [10]:
import glob

EXTEND_IBES_LINK = True

# ---------- 交易日历（逐文件先去重，避免一次读 6000 万行）----------
_days = [pd.read_parquet(p, columns=["dlycaldt"])["dlycaldt"].drop_duplicates()
         for p in sorted(glob.glob(f"{DATA}/crsp_daily_*.parquet"))]
cal = pd.to_datetime(pd.concat(_days)).drop_duplicates().sort_values().reset_index(drop=True)
del _days
print(f"交易日历: {len(cal):,} 天, {cal.min().date()} ~ {cal.max().date()}")

# ---------- IBES actuals ----------
act = pd.read_parquet(f"{DATA}/ibes_actuals.parquet",
                      columns=["ticker", "cusip", "cname", "pends", "anndats", "anntims", "value"])
act["anndats"] = pd.to_datetime(act["anndats"])
act["pends"]   = pd.to_datetime(act["pends"])
act = act.rename(columns={"value": "actual_eps"})
n0 = len(act)

# ---------- 贴 permno ----------
ibl2 = pd.read_parquet(f"{BUILD}/link_ibes.parquet", columns=["ticker", "permno", "sdate", "edate"])
link_max = ibl2["edate"].max()
ibl2["link_extended"] = False
if EXTEND_IBES_LINK:
    at_end = ibl2["edate"] == link_max
    ibl2.loc[at_end, "edate"] = pd.Timestamp("2099-12-31")
    ibl2.loc[at_end, "link_extended"] = True
    print(f"IBES 连接表末期 {link_max.date()} 仍有效的 {at_end.sum():,} 条链接已延长至 2099-12-31")

m = act.merge(ibl2, on="ticker", how="inner")
m = m[(m["anndats"] >= m["sdate"]) & (m["anndats"] <= m["edate"])].drop(columns=["sdate", "edate"])
n1 = m.groupby(["ticker", "anndats"]).ngroups
print(f"[1] 贴上 permno: {len(m):,} 行 / {n1:,} 个唯一事件 (匹配率 {n1/n0:.1%})")

交易日历: 7,673 天, 1996-01-02 ~ 2026-06-30


IBES 连接表末期 2025-12-31 仍有效的 4,386 条链接已延长至 2099-12-31


[1] 贴上 permno: 627,382 行 / 613,272 个唯一事件 (匹配率 82.6%)


In [11]:
# ---------- 限定最终宇宙 + 去重 ----------
uni2 = pd.read_parquet(f"{BUILD}/universe.parquet",
                       columns=["permno", "secinfostartdt", "secinfoenddt", "siccd", "primaryexch", "in_universe"])
uni2 = uni2[uni2["in_universe"]].drop(columns=["in_universe"])
m = m.merge(uni2, on="permno", how="inner")
m = m[(m["anndats"] >= m["secinfostartdt"]) & (m["anndats"] <= m["secinfoenddt"])] \
     .drop(columns=["secinfostartdt", "secinfoenddt"])
print(f"[2] 限定最终宇宙: {len(m):,} 行 / {m.groupby(['ticker','anndats','pends']).ngroups:,} 个事件")

# A: 一事件多 permno
m = m.sort_values(["ticker", "anndats", "permno"]).drop_duplicates(["ticker", "anndats", "pends"], keep="first")
# B: 同一 permno-财季多次公告 → 保留最早
_dupB = m.duplicated(["permno", "pends"], keep=False).sum()
m = m.sort_values("anndats").drop_duplicates(["permno", "pends"], keep="first")
print(f"[3] 同一 permno-财季多次公告 {_dupB:,} 行 → 保留最早, 剩 {len(m):,}")
# C: 同一 permno 同一天多个财季 → 标记保留
m["flag_same_day_multi"] = m.duplicated(["permno", "anndats"], keep=False)
# 补报（一次公告多个财季）全部保留：公司确实披露了，LAG 及其高阶项会吸收滞后效应。
# 但这些行共享同一个 CAR，回归时需按 permno×日期聚类，或用下面这列只留当期。
m["is_latest_pends_on_day"] = m.groupby(["permno", "anndats"])["pends"].transform("max") == m["pends"]
print(f"[4] 同一 permno 同一天多个财季: {m['flag_same_day_multi'].sum():,} 行 (全部保留)")
print(f"    其中当期 {m.loc[m['flag_same_day_multi'], 'is_latest_pends_on_day'].sum():,} 行, "
      f"补报旧财季 {(m['flag_same_day_multi'] & ~m['is_latest_pends_on_day']).sum():,} 行")

# ---------- 映射到交易日 ----------
cal_arr = cal.values
idx = np.searchsorted(cal_arr, m["anndats"].values, side="left")
m = m[idx < len(cal_arr)].copy()
m["td0_idx"] = idx[idx < len(cal_arr)]
m["td0"] = cal_arr[m["td0_idx"].values]
m["ann_on_nontrading"] = m["td0"] != m["anndats"]
# 公告日早于交易日历起点时，searchsorted 会把 td0 落到日历第一天（相隔数月），
# 这类事件的 CAR 窗口与公告无关，必须剔除
m["td0_gap_days"] = (m["td0"] - m["anndats"]).dt.days
# 公告日早于交易日历起点(1996-01-02)的事件，td0 被强行落到日历首日，
# CAR 窗口与公告无关 → 整体剔除，这也让 PEAD 样本起点与新闻语料(1996 起)对齐
m["flag_pre_calendar"] = m["anndats"] < cal.min()
m["flag_td0_gap"] = m["flag_pre_calendar"] | (m["td0_gap_days"] > 7)

# ---------- LAG ----------
# 真正"只因延长链接才存在"的事件 = 公告日晚于连接表末期
m["flag_beyond_link_end"] = m["link_extended"] & (m["anndats"] > link_max)

m["lag"] = (m["anndats"] - m["pends"]).dt.days
m["flag_lag_bad"] = ~m["lag"].between(0, 180)

events = m[["permno", "ticker", "anndats", "td0", "td0_idx", "pends", "actual_eps", "anntims",
            "lag", "flag_lag_bad", "ann_on_nontrading", "td0_gap_days", "flag_td0_gap", "flag_pre_calendar",
            "flag_same_day_multi", "is_latest_pends_on_day",
            "link_extended", "flag_beyond_link_end", "siccd", "primaryexch", "cusip", "cname"]] \
    .rename(columns={"link_extended": "flag_link_extended"}) \
    .sort_values(["anndats", "permno"]).reset_index(drop=True)
events.to_parquet(f"{BUILD}/events.parquet", index=False)

print(f"\n事件表: {len(events):,} 条 | {events['anndats'].min().date()} ~ {events['anndats'].max().date()}"
      f" | {events['permno'].nunique():,} 个 permno")
print(f"  公告落在非交易日(顺延): {events['ann_on_nontrading'].mean():.1%}")
print(f"  公告日早于交易日历起点 {cal.min().date()}，CAR 不可用: {events['flag_td0_gap'].sum():,}")
print(f"  LAG 异常(<0 或 >180 天): {events['flag_lag_bad'].mean():.2%}"
      f" | LAG 中位数 {events.loc[~events['flag_lag_bad'], 'lag'].median():.0f} 天")
print(f"  落在连接表末期({link_max.date()})之后、靠延长才得到的事件: "
      f"{events['flag_beyond_link_end'].sum():,}  ← 只有这些有错配风险")
print(f"  (flag_link_extended={events['flag_link_extended'].sum():,} 是所有'链接至今有效'的事件，绝大多数本就能匹配)")
print("\n每年事件数:")
_y = events.groupby(events["anndats"].dt.year).size()
print(pd.concat([_y.head(4), pd.Series({"...": ""}), _y.tail(4)]).to_string())

[2] 限定最终宇宙: 517,966 行 / 517,966 个事件


[3] 同一 permno-财季多次公告 22 行 → 保留最早, 剩 517,955
[4] 同一 permno 同一天多个财季: 13,118 行 (全部保留)
    其中当期 4,786 行, 补报旧财季 8,332 行



事件表: 517,955 条 | 1995-04-01 ~ 2026-05-14 | 14,214 个 permno
  公告落在非交易日(顺延): 1.6%
  公告日早于交易日历起点 1996-01-02，CAR 不可用: 5,628
  LAG 异常(<0 或 >180 天): 1.15% | LAG 中位数 33 天
  落在连接表末期(2025-12-31)之后、靠延长才得到的事件: 5,846  ← 只有这些有错配风险
  (flag_link_extended=168,515 是所有'链接至今有效'的事件，绝大多数本就能匹配)

每年事件数:
1995     5628
1996    21932
1997    23601
1998    23870
...          
2023    14448
2024    13914
2025    13449
2026     5846


## 4. CAR

$$CAR^{ANN}_{i,d}=\prod_{k=d}^{d+1}(1+R_{i,k})-\prod_{k=d}^{d+1}(1+R_{p,k}),\qquad
CAR^{DRIFT}_{i,d}=\prod_{k=d+2}^{d+61}(1+R_{i,k})-\prod_{k=d+2}^{d+61}(1+R_{p,k})$$

**实现**：不逐事件循环，而是把日收益转成每只股票的**累积对数收益**，窗口收益 = 两端相减取指数。
按交易日流式推进，内存 O(股票数) 与数据量无关（直接读 6000 万行面板会 OOM）。全程约 50 秒。

**四个细节**

1. `ret = −1`（退市全损）会让 `log1p` 变 `−inf` 并污染该股票之后的整条累积序列 → 下限截到 `−0.999999`
2. 窗口内一天交易记录都没有 → CAR 记 `NaN`，不是 0
3. 窗口跨过样本末端（2026 年的事件）→ 用截至末端的累积值，`n_days_*` 会小于满窗，由 `flag_short_window` 标记
4. 个股与基准**必须用同一口径**：C2C 配 C2C 基准，O2O 配 O2O 基准，四个 CAR 列两两配对

In [12]:
RET_FLOOR = -0.999999          # 退市全损 ret=-1 → log1p 为 -inf，会污染整条累积序列

# ---------- 交易日历 ----------
_d = [pd.read_parquet(p, columns=["dlycaldt"])["dlycaldt"].drop_duplicates()
      for p in sorted(glob.glob(f"{DATA}/crsp_daily_*.parquet"))]
cal = pd.to_datetime(pd.concat(_d)).drop_duplicates().sort_values().reset_index(drop=True)
del _d
cal_map = pd.Series(np.arange(len(cal), dtype="int32"), index=cal.values)
NDAY = len(cal)

# ---------- 事件与锚点 ----------
ev = pd.read_parquet(f"{BUILD}/events.parquet", columns=["permno", "td0", "td0_idx"])
ev["permno"] = ev["permno"].astype("int64")
ev = ev.reset_index().rename(columns={"index": "eid"})
PERM = ev["permno"].to_numpy()
ANCH = {"a0": ev["td0_idx"].to_numpy() - 1,
        "a1": ev["td0_idx"].to_numpy() + 1,
        "a2": ev["td0_idx"].to_numpy() + 61}
PMAX = int(PERM.max()) + 1
print(f"交易日历 {NDAY:,} 天 | 事件 {len(ev):,}", flush=True)


def stream_cum(day_iter, anchors, permno_arr, permno_max):
    """按交易日推进，取每个锚点处的 (cumlog, cumcnt)。day_iter 必须按 td_idx 升序、每天完整。"""
    cum = np.zeros(permno_max, dtype="float64")
    cnt = np.zeros(permno_max, dtype="int32")
    out = {k: [np.full(len(v), np.nan), np.zeros(len(v), dtype="int32")] for k, v in anchors.items()}
    todo = {}
    for k, v in anchors.items():
        for pos, idx in enumerate(v):
            if 0 <= idx < NDAY:
                todo.setdefault(int(idx), []).append((k, pos))

    for td_idx, pmn, ret in day_iter:
        np.add.at(cum, pmn, np.log1p(np.maximum(ret, RET_FLOOR)))
        np.add.at(cnt, pmn, 1)
        for k, pos in todo.get(td_idx, ()):
            p = PERM[pos] if permno_arr is None else permno_arr[pos]
            out[k][0][pos] = cum[p]
            out[k][1][pos] = cnt[p]

    for k, v in anchors.items():
        beyond = np.where(v >= NDAY)[0]                       # 窗口超出样本末端 → 用最终值
        for pos in beyond:
            p = permno_arr[pos]
            out[k][0][pos] = cum[p]
            out[k][1][pos] = cnt[p]
        before = np.where(v < 0)[0]                            # 窗口起点早于样本 → 0
        out[k][0][before] = 0.0
        out[k][1][before] = 0
    return out


def daily_c2c():
    for p in sorted(glob.glob(f"{DATA}/crsp_daily_*.parquet")):
        d = pd.read_parquet(p, columns=["permno", "dlycaldt", "dlyret"]).dropna(subset=["dlyret"])
        d["td_idx"] = pd.to_datetime(d["dlycaldt"]).map(cal_map)
        d = d.dropna(subset=["td_idx"])
        d["td_idx"] = d["td_idx"].astype("int32"); d["permno"] = d["permno"].astype("int64")
        for td_idx, g in d.groupby("td_idx", sort=True):
            yield int(td_idx), g["permno"].to_numpy(), g["dlyret"].to_numpy(dtype="float64")


def daily_o2o():
    """export 面板按 (date, PERMNO) 排序；跨 batch 缓冲，保证每天完整后再产出。"""
    pf = pq.ParquetFile(f"{EXPORT}/crsp_daily_ret_c2c_o2o.parquet")
    buf_idx, buf_p, buf_r = None, [], []
    for b in pf.iter_batches(batch_size=2_000_000, columns=["PERMNO", "date", "O2O_RET"]):
        d = b.to_pandas().dropna(subset=["O2O_RET"])
        if d.empty:
            continue
        d["td_idx"] = pd.to_datetime(d["date"]).map(cal_map)
        d = d.dropna(subset=["td_idx"])
        d["td_idx"] = d["td_idx"].astype("int32"); d["PERMNO"] = d["PERMNO"].astype("int64")
        for td_idx, g in d.groupby("td_idx", sort=True):
            td_idx = int(td_idx)
            if buf_idx is None:
                buf_idx = td_idx
            if td_idx != buf_idx:
                yield buf_idx, np.concatenate(buf_p), np.concatenate(buf_r)
                buf_idx, buf_p, buf_r = td_idx, [], []
            buf_p.append(g["PERMNO"].to_numpy())
            buf_r.append(g["O2O_RET"].to_numpy(dtype="float64"))
    if buf_idx is not None:
        yield buf_idx, np.concatenate(buf_p), np.concatenate(buf_r)


def to_car(res, tag):
    ann = np.expm1(res["a1"][0] - res["a0"][0])
    dft = np.expm1(res["a2"][0] - res["a1"][0])
    n_ann = res["a1"][1] - res["a0"][1]
    n_dft = res["a2"][1] - res["a1"][1]
    ann = np.where(n_ann > 0, ann, np.nan)          # 窗口内无交易记录 → 缺失, 不是 0
    dft = np.where(n_dft > 0, dft, np.nan)
    return pd.DataFrame({f"stk_ann_{tag}": ann, f"stk_drift_{tag}": dft,
                         f"n_days_ann_{tag}": n_ann, f"n_days_drift_{tag}": n_dft})


car = ev[["eid", "permno", "td0", "td0_idx"]].copy()
car = pd.concat([car, to_car(stream_cum(daily_c2c(), ANCH, PERM, PMAX), "c2c")], axis=1)
print("C2C 个股 BHR 完成", flush=True)
car = pd.concat([car, to_car(stream_cum(daily_o2o(), ANCH, PERM, PMAX), "o2o")], axis=1)
print("O2O 个股 BHR 完成", flush=True)

# ---------- 基准组合 ----------
mem = pd.read_parquet(f"{BUILD}/port25_membership.parquet", columns=["permno", "ffyear", "port25"])
car["ffyear"] = np.where(car["td0"].dt.month >= 7, car["td0"].dt.year, car["td0"].dt.year - 1)
car = car.merge(mem, on=["permno", "ffyear"], how="left")
print(f"事件贴上 port25: {car['port25'].notna().mean():.1%}", flush=True)

bench = pd.read_parquet(f"{BUILD}/port25_bench_returns.parquet")
bench["td_idx"] = pd.to_datetime(bench["date"]).map(cal_map)
bench = bench.dropna(subset=["td_idx"])
bench["td_idx"] = bench["td_idx"].astype(int)

# 25 × NDAY 的累积对数收益矩阵（组合每天都有值，直接下标取用）
CUMB = {}
for tag, col in [("c2c", "bench_ret_c2c"), ("o2o", "bench_ret_o2o")]:
    mat = np.zeros((26, NDAY), dtype="float64")
    piv = bench.pivot_table(index="port25", columns="td_idx", values=col)
    piv = piv.reindex(columns=range(NDAY))
    vals = np.nan_to_num(piv.to_numpy(dtype="float64"), nan=0.0)
    mat[piv.index.to_numpy().astype(int), :] = np.cumsum(np.log1p(np.maximum(vals, RET_FLOOR)), axis=1)
    CUMB[tag] = mat

p25 = car["port25"].fillna(0).to_numpy().astype(int)
i0 = np.clip(ANCH["a0"], 0, NDAY - 1); i1 = np.clip(ANCH["a1"], 0, NDAY - 1); i2 = np.clip(ANCH["a2"], 0, NDAY - 1)
for tag in ("c2c", "o2o"):
    M = CUMB[tag]
    b_ann = np.expm1(M[p25, i1] - M[p25, i0])
    b_dft = np.expm1(M[p25, i2] - M[p25, i1])
    ok = car["port25"].notna().to_numpy()
    car[f"bench_ann_{tag}"] = np.where(ok, b_ann, np.nan)
    car[f"bench_drift_{tag}"] = np.where(ok, b_dft, np.nan)
    car[f"car_ann_{tag}"] = car[f"stk_ann_{tag}"] - car[f"bench_ann_{tag}"]
    car[f"car_drift_{tag}"] = car[f"stk_drift_{tag}"] - car[f"bench_drift_{tag}"]

# 公告日早于交易日历起点的事件：td0 被强行落到日历首日，窗口与公告无关 → CAR 置空
_gap = pd.read_parquet(f"{BUILD}/events.parquet", columns=["flag_td0_gap"])["flag_td0_gap"].to_numpy()
for _c in [c for c in car.columns if c.startswith(("car_", "stk_", "bench_"))]:
    car.loc[_gap, _c] = np.nan
print(f"因 td0 间隔过大而置空 CAR 的事件: {_gap.sum():,}")

car["flag_short_window"] = (car["n_days_ann_c2c"] < 2) | (car["n_days_drift_c2c"] < 60)
car.drop(columns=["ffyear"]).to_parquet(f"{BUILD}/car.parquet", index=False)

del CUMB, bench, mem
gc.collect()

cols = ["car_ann_c2c", "car_drift_c2c", "car_ann_o2o", "car_drift_o2o"]
print("\nCAR 分布:")
print(car[cols].describe(percentiles=[.01, .25, .5, .75, .99]).round(4).to_string())
print("\n缺失率:", {c: f"{car[c].isna().mean():.1%}" for c in cols})
print("窗口不完整(ANN<2天 或 DRIFT<60天):", f"{car['flag_short_window'].mean():.1%}")
print("\nC2C 与 O2O 的相关: ANN %.4f | DRIFT %.4f" % (
    car["car_ann_c2c"].corr(car["car_ann_o2o"]), car["car_drift_c2c"].corr(car["car_drift_o2o"])))

交易日历 7,673 天 | 事件 517,955


C2C 个股 BHR 完成


O2O 个股 BHR 完成


事件贴上 port25: 88.8%


因 td0 间隔过大而置空 CAR 的事件: 5,628



CAR 分布:
       car_ann_c2c  car_drift_c2c  car_ann_o2o  car_drift_o2o
count  454218.0000    454207.0000  451493.0000    448864.0000
mean       -0.0003        -0.0160       0.0020        -0.0235
std         0.0992         0.3053       0.0887         0.3052
min        -1.0071        -1.7704      -0.8729        -1.7745
1%         -0.2625        -0.6453      -0.2316        -0.6565
25%        -0.0396        -0.1420      -0.0331        -0.1526
50%        -0.0016        -0.0271      -0.0000        -0.0327
75%         0.0366         0.0807       0.0344         0.0773
99%         0.2825         0.8973       0.2552         0.8918
max         4.9180        26.6967       4.7047        26.1401

缺失率: {'car_ann_c2c': '12.3%', 'car_drift_c2c': '12.3%', 'car_ann_o2o': '12.8%', 'car_drift_o2o': '13.3%'}
窗口不完整(ANN<2天 或 DRIFT<60天): 3.3%

C2C 与 O2O 的相关: ANN 0.7317 | DRIFT 0.9602


### 4.1 诊断

In [13]:
carq = pd.read_parquet(f"{BUILD}/car.parquet")
ev_full = pd.read_parquet(f"{BUILD}/events.parquet", columns=["permno", "anndats"]).reset_index(drop=True)
carq["year"] = ev_full["anndats"].dt.year.values

print("port25 覆盖率（缺失 = 该公司当年没有 BE 或 6 月末市值）:")
cov = carq.groupby("year")["port25"].apply(lambda s: s.notna().mean())
print(pd.concat([cov.head(3), pd.Series({"...": np.nan}), cov.tail(3)]).round(3).to_string())

print("\n逐年 CAR 均值 (C2C):")
yr = carq.groupby("year")[["car_ann_c2c", "car_drift_c2c"]].mean()
print(pd.concat([yr.head(3), yr.tail(3)]).round(4).to_string())

print("\n窗口天数分布 (C2C):")
print(f"  ANN   满窗(2天) 占 {(carq['n_days_ann_c2c'] == 2).mean():.1%}")
print(f"  DRIFT 满窗(60天) 占 {(carq['n_days_drift_c2c'] == 60).mean():.1%}")

port25 覆盖率（缺失 = 该公司当年没有 BE 或 6 月末市值）:
1995    0.907
1996    0.868
1997    0.863
...       NaN
2024    0.893
2025    0.881
2026    0.882

逐年 CAR 均值 (C2C):
      car_ann_c2c  car_drift_c2c
year                            
1995          NaN            NaN
1996       0.0012        -0.0358
1997       0.0002        -0.0311
2024      -0.0011        -0.0128
2025      -0.0014        -0.0098
2026      -0.0004        -0.0151

窗口天数分布 (C2C):
  ANN   满窗(2天) 占 98.9%
  DRIFT 满窗(60天) 占 96.7%


## 5. SUE（分析师共识法）

$$SUE_{i,d}=\frac{e_{i,d}-F_{i,d}}{P_{i,d}}$$

| 分量 | 构造 |
|---|---|
| $e$ | `ibes_actuals` 的实际 EPS |
| $F$ | 公告前 **60 个日历天**内 issued(`actdats`) 或 reviewed(`revdats`) 的 **1–2 季度前瞻**预测（`fpi ∈ {6,7}`），同一分析师取最新一条，再取**中位数** |
| $P$ | 财季末股价，**折算到与 EPS 相同的拆股基准**：`\|mthprc\| / mthcumfacpr` |

**为什么 P 必须折算**：I/B/E/S 的 EPS 已统一到最新拆股基准（AAPL 2012Q2 记为 0.4393 而非披露时的 \$12.30，
差 28 倍 = 2014 年 7:1 × 2020 年 4:1），而 CRSP 的 `mthprc` 是当年历史价格。
直接相除会让拆过股的公司 SUE 被系统性缩小，且每家缩小倍数不同，污染截面排序。

**清洗规则**（HLT 2009 Sec II 末尾）：① 剔除 \|e\| 或 \|F\| 大于股价的观测；
② 剔除**拆股调整前**股价 < \$1 的观测（所以 `prc_unadj` 原值也要留）；③ 只用 1–2 季度前瞻预测。

**分十分位**：按公告日所在**日历季度**独立分十份（`sue_dec` ∈ 1..10）。

In [14]:
ev = pd.read_parquet(f"{BUILD}/events.parquet",
                     columns=["permno", "ticker", "anndats", "pends", "actual_eps"]).reset_index()
ev = ev.rename(columns={"index": "eid"})
ev["pyear"] = ev["pends"].dt.year
print(f"事件 {len(ev):,}", flush=True)

# ---------- 逐年算共识预测 ----------
out = []
for yr in sorted(ev["pyear"].unique()):
    f = f"{DATA}/ibes_detail_{yr}.parquet"
    try:
        det = pd.read_parquet(f, columns=["ticker", "analys", "fpedats", "actdats", "revdats", "value", "fpi"])
    except FileNotFoundError:
        continue
    det = det[det["fpi"].isin(["6", "7"])]
    for c in ("fpedats", "actdats", "revdats"):
        det[c] = pd.to_datetime(det[c])

    sub = ev[ev["pyear"] == yr][["eid", "ticker", "anndats", "pends"]]
    m = sub.merge(det, left_on=["ticker", "pends"], right_on=["ticker", "fpedats"], how="inner")
    if m.empty:
        continue
    w0 = m["anndats"] - pd.Timedelta(days=60)
    in_win = (((m["actdats"] >= w0) & (m["actdats"] < m["anndats"])) |
              ((m["revdats"] >= w0) & (m["revdats"] < m["anndats"])))
    m = m[in_win]
    if m.empty:
        continue
    # 同一分析师取最新一条。排序日期必须是**公告前**最近一次活动日：
    # revdats（复核日）常远在公告之后（23% 的事件如此，中位数晚 167 天），
    # 用 max(actdats, revdats) 排序会让公告后的复核日决定"谁最新"——既有前视，
    # 也不是 HLT 说的最新一条。取两者中落在公告前的较晚者。
    _a = m["actdats"].where(m["actdats"] < m["anndats"])
    _r = m["revdats"].where(m["revdats"] < m["anndats"])
    m["last_dt"] = np.fmax(_a.astype("int64").where(_a.notna()),
                           _r.astype("int64").where(_r.notna()))
    m["last_dt"] = pd.to_datetime(m["last_dt"])
    m = m.sort_values("last_dt").drop_duplicates(["eid", "analys"], keep="last")
    g = m.groupby("eid").agg(consensus_f=("value", "median"),
                             n_analyst_sue=("analys", "nunique"),
                             fcst_last_dt=("last_dt", "max"))
    out.append(g.reset_index())
    print(f"  {yr}: {len(g):,} 个事件有共识", flush=True)

cons = pd.concat(out, ignore_index=True)
print(f"\n有共识预测的事件: {len(cons):,} ({len(cons)/len(ev):.1%})", flush=True)

# ---------- 财季末股价 ----------
mth = pd.read_parquet(f"{DATA}/crsp_monthly.parquet",
                      columns=["permno", "mthcaldt", "mthprc", "mthcumfacpr"])
mth["mthcaldt"] = pd.to_datetime(mth["mthcaldt"])
mth["ym"] = mth["mthcaldt"].dt.to_period("M")
mth["prc_unadj"] = mth["mthprc"].abs()
mth["price_adj"] = mth["prc_unadj"] / mth["mthcumfacpr"].where(mth["mthcumfacpr"] > 0)
px = mth[["permno", "ym", "prc_unadj", "price_adj"]]

sue = ev.merge(cons, on="eid", how="left")
sue["ym"] = sue["pends"].dt.to_period("M")
sue = sue.merge(px, on=["permno", "ym"], how="left")
print(f"能取到财季末股价的事件: {sue['price_adj'].notna().mean():.1%}")

# ---------- SUE + 清洗 ----------
sue["sue"] = (sue["actual_eps"] - sue["consensus_f"]) / sue["price_adj"]

flag_price = sue["prc_unadj"] < 1.0                                   # 拆股调整前股价 < $1
flag_big = (sue["actual_eps"].abs() > sue["price_adj"]) | (sue["consensus_f"].abs() > sue["price_adj"])
sue["flag_sue_dropped"] = flag_price | flag_big
sue.loc[sue["flag_sue_dropped"], "sue"] = np.nan
print(f"清洗: 股价<$1 剔除 {flag_price.sum():,} | |e| 或 |F| > 股价 剔除 {flag_big.sum():,}")
print(f"最终有 SUE 的事件: {sue['sue'].notna().sum():,} ({sue['sue'].notna().mean():.1%})")

# ---------- 按日历季度分十分位 ----------
sue["qtr"] = sue["anndats"].dt.to_period("Q")
# 注意：sue 是 nullable Float64，pd.qcut 对 pd.NA 的处理与 np.nan 不同，
# 会把缺失值也打进极端分位。必须先转成 float64，再显式把缺失位置置回 NaN。
sue["_sue_f64"] = pd.to_numeric(sue["sue"], errors="coerce").astype("float64")
sue["sue_dec"] = (sue.groupby("qtr")["_sue_f64"]
                     .transform(lambda x: pd.qcut(x.rank(method="first"), 10, labels=False) + 1
                                if x.notna().sum() >= 100 else np.nan))
sue.loc[sue["_sue_f64"].isna(), "sue_dec"] = np.nan
_s = sue.pop("_sue_f64")
assert not (sue["sue_dec"].notna() & _s.isna()).any(), "sue_dec 不应出现在 sue 缺失的行上"

cols = ["eid", "permno", "anndats", "pends", "actual_eps", "consensus_f", "n_analyst_sue",
        "fcst_last_dt", "price_adj", "prc_unadj", "sue", "sue_dec", "flag_sue_dropped"]
sue[cols].to_parquet(f"{BUILD}/sue.parquet", index=False)

del mth, px, cons, out
gc.collect()

print("\nSUE 分布:")
print(sue["sue"].describe(percentiles=[.01, .25, .5, .75, .99]).round(5).to_string())
print("\n分析师人数分布:", sue["n_analyst_sue"].describe(percentiles=[.25, .5, .75]).round(1).to_dict())
print("\n各十分位的 SUE 均值:")
print(sue.groupby("sue_dec")["sue"].agg(["mean", "count"]).round(5).to_string())

事件 517,955


  1995: 5,644 个事件有共识


  1996: 11,987 个事件有共识


  1997: 12,513 个事件有共识


  1998: 12,312 个事件有共识


...

  2025: 10,675 个事件有共识


  2026: 2,599 个事件有共识



有共识预测的事件: 353,842 (68.3%)


能取到财季末股价的事件: 98.5%
清洗: 股价<$1 剔除 18,454 | |e| 或 |F| > 股价 剔除 7,895
最终有 SUE 的事件: 344,026 (66.4%)



SUE 分布:
count    344026.0
mean     -0.00074
std       0.03673
min      -1.52893
1%       -0.09472
25%      -0.00072
50%        0.0005
75%       0.00244
99%       0.05882
max       1.78605

分析师人数分布: {'count': 353842.0, 'mean': 5.0, 'std': 4.8, 'min': 1.0, '25%': 2.0, '50%': 3.0, '75%': 7.0, 'max': 44.0}

各十分位的 SUE 均值:
            mean  count
sue_dec                
1.0     -0.04307  34458
2.0      -0.0038  34394
3.0     -0.00089  34387
4.0     -0.00003  34397
5.0      0.00034  34411
6.0      0.00082  34365
7.0       0.0015  34379
8.0      0.00265  34405
9.0      0.00518  34382
10.0     0.02995  34448


### 5.1 验证：PEAD 是否复制出来

把 CAR 与 SUE 合并，看各 SUE 十分位的平均 CAR。若整条流水线正确，应看到：

- `CAR^ANN` 随 SUE 十分位**单调递增**（盈余意外越大，公告当下反应越强）
- `CAR^DRIFT` 同样**单调递增**（这就是 PEAD：反应不完全，剩余部分在之后 60 个交易日继续漂移）

In [15]:
car_v = pd.read_parquet(f"{BUILD}/car.parquet")
sue_v = pd.read_parquet(f"{BUILD}/sue.parquet", columns=["eid", "sue", "sue_dec", "n_analyst_sue"])
chk = car_v.merge(sue_v, on="eid", how="inner")
chk = chk[chk["sue_dec"].notna()]
CARC = ["car_ann_c2c", "car_drift_c2c", "car_ann_o2o", "car_drift_o2o"]

tab = chk.groupby("sue_dec")[CARC].mean() * 100
tab["n"] = chk.groupby("sue_dec").size()
print(f"样本 {len(chk):,}\n各 SUE 十分位的平均 CAR (%):")
print(tab.round(3).to_string())


def welch(x, y):
    x, y = x.dropna().to_numpy(), y.dropna().to_numpy()
    se = np.sqrt(x.var(ddof=1) / len(x) + y.var(ddof=1) / len(y))
    return x.mean() - y.mean(), (x.mean() - y.mean()) / se


print("\nD10 − D1:")
for c in CARC:
    diff, t = welch(chk.loc[chk["sue_dec"] == 10, c], chk.loc[chk["sue_dec"] == 1, c])
    mono = (tab[c].diff().dropna() > 0).all()
    print(f"  {c:16s} {100*diff:+.2f}%   t = {t:6.1f}   十分位单调递增: {mono}")

样本 344,026
各 SUE 十分位的平均 CAR (%):
         car_ann_c2c  car_drift_c2c  car_ann_o2o  car_drift_o2o      n
sue_dec                                                               
1.0           -4.164         -3.541       -3.589         -5.037  34458
2.0           -3.038         -1.507       -2.735         -2.462  34394
3.0           -2.109         -1.424       -1.970         -1.994  34387
4.0           -1.221         -1.188       -1.048         -1.789  34397
5.0           -0.160         -0.894       -0.114         -1.179  34411
6.0            0.718         -0.648        0.723         -0.852  34365
7.0            1.471         -0.612        1.380         -0.835  34379
8.0            2.095         -0.179        1.999         -0.462  34405
9.0            2.890         -0.041        2.745         -0.320  34382
10.0           3.920          0.692        3.701          0.366  34448

D10 − D1:
  car_ann_c2c      +8.08%   t =   82.6   十分位单调递增: True
  car_drift_c2c    +4.23%   t =   14.5   十分位单调递增:

## 6. 控制变量（HLT 2009 Sec IV.A.1）

| 变量 | 粒度 | 构造 |
|---|---|---|
| `size_dec` | firm×year | 年初（上年 12 月末）市值，NYSE breakpoint 分十位 |
| `bm_dec` | firm×year | 与 25 组同一套 BE/ME，NYSE breakpoint 独立分十位 |
| `lnanalyst` | firm×event | log(1 + 公告前 365 天内出过预测的不同分析师数) |
| `lag` / `lag2` / `lag3` | firm×event | `anndats − pends` 及其高阶项 |
| `io` | firm×quarter | 13F Σshares / 流通股，取公告前最近一期报告期 |
| `evol` | firm×event | 过去 16 季**季节差分** EPS 的标准差（≥4 个观测） |
| `epersist` | firm×event | 同一序列的一阶自相关 |
| `turn` | firm×event | 过去 12 个月的月均换手率 |

**三个实现细节**

1. **EPS 要除以 `ajexq`** 才能跨季比较，但 `ajexq` 有 4.1% 的行小于 0.01（甚至为 0），
   除下去会炸出 −34 亿这种值。加下限 `ajexq ≥ 0.01`，这些行剔除。
2. **TURN 不需要拆股调整**：`mthvol` 与 `shrout` 是同月口径，比值本身自洽
   （AAPL 2015-06 月换手率 0.154）。单位：`mthvol` 是股数，`shrout` 是千股。
3. **IO 的单位同理**：13F `shares` 是股数，除以 `shrout × 1000`
   （AAPL 2015Q2 = 0.586，与实际机构持股比例吻合）。超过两个季度未更新的记为缺失。

In [16]:
AJEX_MIN = 0.01   # ajexq 下限，低于此值 EPS 折算会数值爆炸


def nyse_decile(g, col):
    """用 NYSE 股票算十分位断点，再给全市场打标签。"""
    nyse = g.loc[g["is_nyse"], col].dropna()
    if len(nyse) < 50:
        return pd.Series(np.nan, index=g.index)
    bps = np.percentile(nyse, np.arange(10, 100, 10))
    return pd.Series(np.searchsorted(bps, g[col], side="right") + 1.0, index=g.index)


mth_c = pd.read_parquet(f"{DATA}/crsp_monthly.parquet",
                        columns=["permno", "mthcaldt", "mthcap", "shrout", "mthvol"])
mth_c["mthcaldt"] = pd.to_datetime(mth_c["mthcaldt"])
mth_c["ym"] = mth_c["mthcaldt"].dt.to_period("M")
uni_flag = universe[universe["in_universe"]].drop_duplicates("permno").set_index("permno")["is_nyse"]

# ---- SIZE：年初(上年12月末)市值 ----
dec = mth_c[mth_c["mthcaldt"].dt.month == 12][["permno", "mthcaldt", "mthcap"]].copy()
dec["year"] = dec["mthcaldt"].dt.year + 1
dec = dec[dec["mthcap"] > 0]
dec["is_nyse"] = dec["permno"].map(uni_flag)
dec["size_dec"] = dec.groupby("year", group_keys=False)[["mthcap", "is_nyse"]].apply(lambda g: nyse_decile(g, "mthcap"))
ctrl_size = dec[["permno", "year", "mthcap", "size_dec"]].rename(columns={"mthcap": "me_jan"})

# ---- BM：与 25 组同一套 BE/ME ----
mem_c = pd.read_parquet(f"{BUILD}/port25_membership.parquet",
                        columns=["permno", "ffyear", "bm_raw", "be", "is_nyse"])
mem_c["bm_dec"] = mem_c.groupby("ffyear", group_keys=False)[["bm_raw", "is_nyse"]].apply(lambda g: nyse_decile(g, "bm_raw"))
ctrl_bm = mem_c[["permno", "ffyear", "bm_raw", "be", "bm_dec"]]

# ---- TURN：过去 12 个月月均换手率 ----
tn = mth_c[["permno", "ym", "mthvol", "shrout"]].copy()
tn["turn_m"] = tn["mthvol"] / (tn["shrout"] * 1000.0)
tn = tn.dropna(subset=["turn_m"]).sort_values(["permno", "ym"])
tn["turn"] = tn.groupby("permno")["turn_m"].transform(lambda s: s.rolling(12, min_periods=6).mean().shift(1))
ctrl_turn = tn[["permno", "ym", "turn"]].dropna()
print(f"SIZE {len(ctrl_size):,} | BM {len(ctrl_bm):,} | TURN {len(ctrl_turn):,}", flush=True)

# ---- IO：13F（1.14 亿行，逐年聚合）----
IO_PATH = f"{BUILD}/ctrl_io.parquet"
if os.path.exists(IO_PATH) and not FORCE:
    ctrl_io = pd.read_parquet(IO_PATH)
else:
    sec = pd.read_parquet(f"{DATA}/crsp_security_info.parquet",
                          columns=["permno", "cusip", "secinfostartdt", "secinfoenddt"])
    sec["secinfostartdt"] = pd.to_datetime(sec["secinfostartdt"]); sec["secinfoenddt"] = pd.to_datetime(sec["secinfoenddt"])
    parts = []
    for p in sorted(glob.glob(f"{DATA}/tr13f_*.parquet")):
        t = pd.read_parquet(p, columns=["cusip", "rdate", "fdate", "mgrno", "shares"])
        if t.empty:
            continue
        t["rdate"] = pd.to_datetime(t["rdate"]); t["fdate"] = pd.to_datetime(t["fdate"])
        t = t.dropna(subset=["shares", "cusip"])
        t = t.sort_values("fdate").drop_duplicates(["mgrno", "cusip", "rdate"], keep="last")   # 修正申报取最新
        parts.append(t.groupby(["cusip", "rdate"], as_index=False)["shares"].sum())
    io = pd.concat(parts, ignore_index=True).groupby(["cusip", "rdate"], as_index=False)["shares"].sum()
    io = io.merge(sec, on="cusip", how="inner")
    io = io[(io["rdate"] >= io["secinfostartdt"]) & (io["rdate"] <= io["secinfoenddt"])]
    io["ym"] = io["rdate"].dt.to_period("M")
    io = io.merge(mth_c[["permno", "ym", "shrout"]], on=["permno", "ym"], how="inner")
    io["io"] = io["shares"] / (io["shrout"] * 1000.0)
    io = io[(io["io"] > 0) & (io["io"] < 2)]
    ctrl_io = io[["permno", "rdate", "io"]].sort_values(["permno", "rdate"])
    ctrl_io.to_parquet(IO_PATH, index=False)
print(f"IO {len(ctrl_io):,} 条, 中位数 {ctrl_io['io'].median():.3f}", flush=True)

# ---- EVOL / EPERSIST：过去 16 季季节差分 EPS ----
fq = pd.read_parquet(f"{DATA}/compustat_quarterly.parquet",
                     columns=["gvkey", "datadate", "fyearq", "fqtr", "epspxq", "ajexq"])
fq["datadate"] = pd.to_datetime(fq["datadate"])
fq = fq.dropna(subset=["epspxq", "fyearq", "fqtr"])
bad = fq["ajexq"].isna() | (fq["ajexq"] < AJEX_MIN)
print(f"ajexq < {AJEX_MIN} 或缺失被剔除: {bad.sum():,} 行 ({bad.mean():.2%})")
fq = fq[~bad].copy()
fq["eps_adj"] = fq["epspxq"] / fq["ajexq"]
fq = fq.sort_values(["gvkey", "datadate"]).drop_duplicates(["gvkey", "fyearq", "fqtr"], keep="last")
fq["qidx"] = (fq["fyearq"] * 4 + fq["fqtr"]).astype(int)
_prev = fq[["gvkey", "qidx", "eps_adj"]].copy(); _prev["qidx"] += 4
fq = fq.merge(_prev.rename(columns={"eps_adj": "eps_lag4"}), on=["gvkey", "qidx"], how="left")
fq["dseps"] = fq["eps_adj"] - fq["eps_lag4"]

# 一阶自相关用滚动和向量化（rolling.apply 慢 10 倍）
fq["_y"] = fq.groupby("gvkey")["dseps"].shift(1)
fq["_ok"] = (fq["dseps"].notna() & fq["_y"].notna()).astype(float)
fq["_sx"] = fq["dseps"].where(fq["_ok"] > 0); fq["_sy"] = fq["_y"].where(fq["_ok"] > 0)
fq["_sxy"] = fq["_sx"] * fq["_sy"]; fq["_sxx"] = fq["_sx"] ** 2; fq["_syy"] = fq["_sy"] ** 2
_r = lambda c: fq.groupby("gvkey")[c].transform(lambda z: z.rolling(16, min_periods=4).sum())
_n, _sx, _sy, _sxy, _sxx, _syy = (_r(c) for c in ["_ok", "_sx", "_sy", "_sxy", "_sxx", "_syy"])
_cov = _sxy / _n - (_sx / _n) * (_sy / _n)
_vx = _sxx / _n - (_sx / _n) ** 2; _vy = _syy / _n - (_sy / _n) ** 2
# 窗口内方差为 0（16 个差分全同）会除出 inf；相关系数也必须落在 [−1,1]
_den = np.sqrt(_vx * _vy)
fq["epersist"] = np.where((_n >= 4) & (_den > 0), _cov / _den, np.nan)
fq.loc[fq["epersist"].abs() > 1, "epersist"] = np.nan
fq["evol"] = fq.groupby("gvkey")["dseps"].transform(lambda z: z.rolling(16, min_periods=4).std())

ee = fq[["gvkey", "datadate", "evol", "epersist"]].dropna(subset=["evol"]).merge(ccm, on="gvkey", how="inner")
ee = ee[(ee["datadate"] >= ee["linkdt"]) & (ee["datadate"] <= ee["linkenddt"])]
ee = ee.sort_values("linkprim").drop_duplicates(["gvkey", "datadate"], keep="first")
ctrl_ee = ee[["permno", "datadate", "evol", "epersist"]].sort_values(["permno", "datadate"])
print(f"EVOL/EPERSIST {len(ctrl_ee):,} 条 | EVOL 中位数 {ctrl_ee['evol'].median():.3f}", flush=True)

fq = fq.drop(columns=[c for c in fq.columns if c.startswith("_")])
del dec, tn, mem_c, ee, _n, _sx, _sy, _sxy, _sxx, _syy, _cov, _vx, _vy
gc.collect()

for _n2, _t in [("size", ctrl_size), ("bm", ctrl_bm), ("turn", ctrl_turn), ("evol_epersist", ctrl_ee)]:
    _t.to_parquet(f"{BUILD}/ctrl_{_n2}.parquet", index=False)
del fq
gc.collect()

SIZE 250,874 | BM 135,230 | TURN 2,871,577


IO 889,973 条, 中位数 0.420


ajexq < 0.01 或缺失被剔除: 49,851 行 (4.11%)


/home/zan1/envs/nlp3/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


EVOL/EPERSIST 698,171 条 | EVOL 中位数 0.281


0

### 6.1 LNANALYST

公告前 365 天内出过预测的不同分析师数。逐年处理：某年事件的窗口可能跨到上一年的
`fpedats` 文件，所以每年读 `y−1` 和 `y` 两个文件。无覆盖记录的事件 `lnanalyst = log(1+0) = 0`。

In [17]:
LN_PATH = f"{BUILD}/ctrl_lnanalyst.parquet"
if os.path.exists(LN_PATH) and not FORCE:
    ctrl_ln = pd.read_parquet(LN_PATH)
else:
    events["year"] = events["anndats"].dt.year
    parts = []
    for yr in sorted(events["year"].unique()):
        sub = events.loc[events["year"] == yr, ["anndats", "ticker"]].reset_index().rename(columns={"index": "eid"})
        dets = []
        for y in (yr - 1, yr):
            f = f"{DATA}/ibes_detail_{y}.parquet"
            if os.path.exists(f):
                d = pd.read_parquet(f, columns=["ticker", "analys", "actdats"])
                d["actdats"] = pd.to_datetime(d["actdats"])
                dets.append(d[(d["actdats"] >= pd.Timestamp(f"{yr-1}-01-01")) &
                              (d["actdats"] <= pd.Timestamp(f"{yr}-12-31"))].drop_duplicates())
        if not dets:
            continue
        m = sub.merge(pd.concat(dets, ignore_index=True).drop_duplicates(), on="ticker", how="inner")
        m = m[(m["actdats"] >= m["anndats"] - pd.Timedelta(days=365)) & (m["actdats"] < m["anndats"])]
        parts.append(m.groupby("eid")["analys"].nunique().rename("n_analyst_cover").reset_index())
    ctrl_ln = pd.concat(parts, ignore_index=True)
    ctrl_ln.to_parquet(LN_PATH, index=False)
print(f"LNANALYST: {len(ctrl_ln):,} 个事件有覆盖记录 ({len(ctrl_ln)/len(events):.1%})")

LNANALYST: 443,648 个事件有覆盖记录 (85.7%)


## 7. 合并成事件级大表

`IO` / `EVOL` / `EPERSIST` 用 `merge_asof` 取公告前最近一期，并记录数据的陈旧程度
（`io_stale_days` / `eps_stale_days`），超过阈值视为缺失——否则会把两年前的机构持股当成当期值。

In [18]:
del mth_c, carq, chk, car_v, sue_v, cal_arr, idx
gc.collect()

panel = events.reset_index().rename(columns={"index": "eid"})
panel["year"] = panel["anndats"].dt.year
panel["ffyear"] = np.where(panel["anndats"].dt.month >= 7, panel["year"], panel["year"] - 1)
panel["permno"] = panel["permno"].astype("int64")

panel = panel.merge(pd.read_parquet(f"{BUILD}/car.parquet").drop(columns=["permno", "td0", "td0_idx"]), on="eid", how="left")
panel = panel.merge(pd.read_parquet(f"{BUILD}/sue.parquet").drop(columns=["permno", "anndats", "pends", "actual_eps"]),
                    on="eid", how="left")
panel = panel.merge(ctrl_ln, on="eid", how="left")
panel["lnanalyst"] = np.log1p(panel["n_analyst_cover"].fillna(0))
panel = panel.merge(ctrl_size, on=["permno", "year"], how="left")
panel = panel.merge(ctrl_bm, on=["permno", "ffyear"], how="left")
panel["ym_ann"] = panel["anndats"].dt.to_period("M")
panel = panel.merge(ctrl_turn.rename(columns={"ym": "ym_ann"}), on=["permno", "ym_ann"], how="left")

panel = panel.sort_values("anndats")
_io = ctrl_io.copy(); _io["permno"] = _io["permno"].astype("int64")
panel = pd.merge_asof(panel, _io.sort_values("rdate"), left_on="anndats", right_on="rdate",
                      by="permno", direction="backward")
panel["io_stale_days"] = (panel["anndats"] - panel["rdate"]).dt.days
panel.loc[panel["io_stale_days"] > 200, "io"] = np.nan

_ee = ctrl_ee.copy(); _ee["permno"] = _ee["permno"].astype("int64")
panel = pd.merge_asof(panel, _ee.sort_values("datadate"), left_on="anndats", right_on="datadate",
                      by="permno", direction="backward")
panel["eps_stale_days"] = (panel["anndats"] - panel["datadate"]).dt.days
panel.loc[panel["eps_stale_days"] > 400, ["evol", "epersist"]] = np.nan

panel["lag2"] = panel["lag"] ** 2
panel["lag3"] = panel["lag"] ** 3
panel["month"] = panel["anndats"].dt.month
panel["dow"] = panel["anndats"].dt.dayofweek + 1
panel["qtr"] = panel["anndats"].dt.to_period("Q").astype(str)

panel = panel.sort_values(["anndats", "permno"]).reset_index(drop=True)
panel.to_parquet(f"{BUILD}/pead_panel.parquet", index=False)
print(f"pead_panel: {len(panel):,} 行 × {panel.shape[1]} 列 → {BUILD}/pead_panel.parquet")

CTRL = ["size_dec", "bm_dec", "lnanalyst", "lag", "io", "evol", "epersist", "turn"]
print("\n控制变量覆盖率:")
for c in CTRL:
    print(f"  {c:12s} {panel[c].notna().mean():6.1%}   中位数 {panel[c].median():.4f}")
need = ["car_ann_c2c", "car_drift_c2c", "sue"] + CTRL
print(f"\n关键变量全齐（可进 baseline 回归）: {panel[need].notna().all(axis=1).sum():,} 条 "
      f"({panel[need].notna().all(axis=1).mean():.1%})")

pead_panel: 517,955 行 × 72 列 → build/pead_panel.parquet

控制变量覆盖率:
  size_dec      96.8%   中位数 2.0000
  bm_dec        88.8%   中位数 5.0000
  lnanalyst    100.0%   中位数 1.7918
  lag          100.0%   中位数 33.0000
  io            96.9%   中位数 0.5245
  evol          93.9%   中位数 0.2823
  epersist      92.7%   中位数 0.2008
  turn          96.9%   中位数 0.1151

关键变量全齐（可进 baseline 回归）: 302,564 条 (58.4%)


In [19]:
import pandas as pd
df = pd.read_parquet("build/pead_panel.parquet")
df.head()

,eid,permno,ticker,anndats,td0,td0_idx,pends,actual_eps,anntims,lag,flag_lag_bad,ann_on_nontrading,td0_gap_days,flag_td0_gap,flag_pre_calendar,flag_same_day_multi,is_latest_pends_on_day,flag_link_extended,flag_beyond_link_end,siccd,primaryexch,cusip,cname,year,ffyear,...,prc_unadj,sue,sue_dec,flag_sue_dropped,n_analyst_cover,lnanalyst,me_jan,size_dec,bm_raw,be,bm_dec,ym_ann,turn,rdate,io,io_stale_days,datadate,evol,epersist,eps_stale_days,lag2,lag3,month,dow,qtr
0,0,75554,SEM1,1995-04-01,1996-01-02,0,1995-12-31,-0.15,11:25:00,-274,True,True,276,True,True,False,True,False,False,5060,Q,01655710,ALL AM SEMICOND,1995,1994,...,2.3125,<NA>,NaN,<NA>,2.0,1.098612,23244.38,1.0,<NA>,<NA>,NaN,1995-04,0.177053,NaT,<NA>,NaN,1995-03-31,0.215703,0.263960,1.0,75076,-20570824,4,6,1995Q2
1,1,16791,BKNT,1995-06-09,1996-01-02,0,1995-09-30,<NA>,00:00:00,-113,True,True,207,True,True,False,True,False,False,5621,A,06627910,BANKERS NOTE INC,1995,1994,...,0.5625,<NA>,NaN,True,NaN,0.000000,2253.94,1.0,<NA>,<NA>,NaN,1995-06,0.007273,NaT,<NA>,NaN,1995-04-30,0.031552,-0.292582,40.0,12769,-1442897,6,5,1995Q2
2,2,79713,MAXM,1995-08-02,1996-01-02,0,1995-07-31,0.14,00:00:00,2,False,True,153,True,True,False,True,False,False,5710,Q,57772J10,MAXIM GROUP,1995,1995,...,10.5,<NA>,NaN,<NA>,3.0,1.386294,88412.88,2.0,0.122595,10.839,1.0,1995-08,0.109304,NaT,<NA>,NaN,1995-06-30,0.078102,0.128832,33.0,4,8,8,3,1995Q3
3,3,64697,PLFC,1995-08-04,1996-01-02,0,1995-07-31,-0.21,00:00:00,4,False,True,151,True,True,False,True,False,False,2511,Q,74555310,PULASKI FURN,1995,1995,...,17.5,-0.012,1.0,False,3.0,1.386294,45584.0,1.0,1.2534,57.135,9.0,1995-08,0.043281,NaT,<NA>,NaN,1995-07-31,0.275292,-0.225416,4.0,16,64,8,5,1995Q3
4,4,13100,MA,1995-08-07,1996-01-02,0,1995-07-31,0.3533,00:00:00,7,False,True,148,True,True,False,True,False,False,5311,N,57777810,MAY DEPT STORES,1995,1995,...,43.375,0.000251,6.0,False,31.0,3.465736,8378842.5,10.0,0.435024,3645.0,4.0,1995-08,0.047460,1995-06-30,0.000082,38.0,1995-07-31,0.364214,-0.439078,7.0,49,343,8,1,1995Q3


In [20]:
df.tail()

,eid,permno,ticker,anndats,td0,td0_idx,pends,actual_eps,anntims,lag,flag_lag_bad,ann_on_nontrading,td0_gap_days,flag_td0_gap,flag_pre_calendar,flag_same_day_multi,is_latest_pends_on_day,flag_link_extended,flag_beyond_link_end,siccd,primaryexch,cusip,cname,year,ffyear,...,prc_unadj,sue,sue_dec,flag_sue_dropped,n_analyst_cover,lnanalyst,me_jan,size_dec,bm_raw,be,bm_dec,ym_ann,turn,rdate,io,io_stale_days,datadate,evol,epersist,eps_stale_days,lag2,lag3,month,dow,qtr
517950,517950,91672,MNOV,2026-05-14,2026-05-14,7641,2026-03-31,-0.05,16:17:00,44,False,False,0,False,False,False,True,True,True,2834,Q,58468P20,MEDICINOVA INC,2026,2025,...,1.37,0.014599,10.0,False,4.0,1.609438,64479.51,1.0,0.511726,52.706,6.0,2026-05,0.119512,2025-12-31,0.106486,134.0,2026-03-31,0.023726,0.217730,44.0,1936,85184,5,4,2026Q2
517951,517951,92102,SCOR,2026-05-14,2026-05-14,7641,2026-03-31,-0.39,16:00:00,44,False,False,0,False,False,False,True,True,True,7372,Q,20564W20,COMSCORE,2026,2025,...,6.94,-0.097983,1.0,False,1.0,0.693147,96694.0,1.0,<NA>,<NA>,NaN,2026-05,0.095234,2025-12-31,0.309345,134.0,2026-03-31,7.508186,-0.355043,44.0,1936,85184,5,4,2026Q2
517952,517952,92468,ASOW,2026-05-14,2026-05-14,7641,2026-03-31,-0.55,16:04:00,44,False,False,0,False,False,False,True,True,True,2836,Q,72942G20,PLURI,2026,2025,...,3.395,<NA>,NaN,<NA>,1.0,0.693147,29834.22,1.0,0.004035,0.096,1.0,2026-05,0.049091,2025-12-31,0.002599,134.0,2026-03-31,0.497955,0.709972,44.0,1936,85184,5,4,2026Q2
517953,517953,92557,YPNT,2026-05-14,2026-05-14,7641,2026-03-31,<NA>,11:35:00,44,False,False,0,False,False,False,True,True,True,7313,Q,53814230,LIVE VENTURES,2026,2025,...,11.95,<NA>,NaN,<NA>,NaN,0.000000,45465.6,1.0,2.72225,79.142,10.0,2026-05,0.195389,2025-12-31,0.070623,134.0,2026-03-31,3.784688,0.465219,44.0,1936,85184,5,4,2026Q2
517954,517954,93371,CRMD,2026-05-14,2026-05-14,7641,2026-03-31,0.49,07:30:00,44,False,False,0,False,False,False,True,True,True,2834,Q,21900C30,CORMEDIX,2026,2025,...,6.79,0.022091,10.0,False,7.0,2.079442,921805.43,3.0,0.137612,71.796,2.0,2026-05,0.697539,2025-12-31,0.529256,134.0,2026-03-31,0.405869,0.320832,44.0,1936,85184,5,4,2026Q2


## 8. 数据字典

逐列说明 `build/pead_panel.parquet` 的全部字段，并输出 `build/data_dictionary.csv`
（含每列的 dtype、覆盖率、中位数），供 `回归准备.ipynb` 选列时查阅。

In [21]:
# Step 8 之前释放所有大 DataFrame，否则 kernel 会被 OOM 杀掉
# （前面 7 步累积了 20 多个中间表，此处全部丢弃，字典从磁盘重新读取）
for _n, _o in list(globals().items()):
    if not _n.startswith("_") and isinstance(_o, pd.DataFrame) and len(_o) > 50_000:
        globals().pop(_n, None)
gc.collect()

DICT = {
    # ---- 键与事件信息 ----
    "eid": "事件唯一编号（= events.parquet 的行号，用于回溯各中间表）",
    "permno": "PERMNO (CRSP Permanent Number) CRSP 股票永久标识 —— 本表主键",
    "ticker": "I/B/E/S Ticker：I/B/E/S 内部永久标识（≤6 位字母），不是交易所代码（那是 oftic）",
    "cusip": "CUSIP (Committee on Uniform Securities Identification Procedures) 8 位证券识别码，来自 I/B/E/S",
    "cname": "公司名（I/B/E/S）",
    "anndats": "Announcement Date 盈余公告日 —— 事件锚点",
    "anntims": "Announcement Time 公告时间（可判断盘前/盘后）",
    "td0": "事件日 = 公告日当天或之后的第一个交易日",
    "td0_idx": "td0 在交易日历中的下标（CAR 窗口用它取值）",
    "pends": "Period End Date 财季结束日",
    "actual_eps": "实际 EPS (Earnings Per Share)，I/B/E/S，已折算到最新拆股基准",
    "siccd": "SICCD (Standard Industrial Classification Code) SIC 行业代码，映射 FF10 行业固定效应用",
    "primaryexch": "Primary Exchange 主交易所：N=NYSE / A=AMEX / Q=NASDAQ",
    "year": "公告年", "month": "公告月（1-12）", "dow": "星期几（1=周一）",
    "qtr": "公告所在日历季度（SUE 分位的分组键）",
    "ffyear": "Fama-French formation year 组合形成年（7 月至次年 6 月归属同一年）",
    "ym_ann": "公告所在年月（TURN 的匹配键）",
    # ---- 被解释变量 ----
    "car_ann_c2c": "★ CAR (Cumulative Abnormal Return) 累计异常收益，ANN (Announcement) 窗口 [0,1]，C2C (Close-to-Close) 口径 —— 主口径",
    "car_drift_c2c": "★ CAR 漂移窗口 [2,61]，DRIFT (Post-Earnings-Announcement Drift)，C2C 口径 —— 主口径",
    "car_ann_o2o": "★ CAR 公告窗口 [0,1]，O2O (Open-to-Open) 口径",
    "car_drift_o2o": "★ CAR 漂移窗口 [2,61]，O2O (Open-to-Open) 口径",
    "stk_ann_c2c": "个股 buy-and-hold 收益 [0,1]，C2C (Close-to-Close)（未减基准）",
    "stk_drift_c2c": "个股 buy-and-hold 收益 [2,61]，C2C",
    "stk_ann_o2o": "个股 buy-and-hold 收益 [0,1]，O2O (Open-to-Open)",
    "stk_drift_o2o": "个股 buy-and-hold 收益 [2,61]，O2O",
    "bench_ann_c2c": "基准组合 buy-and-hold 收益 [0,1]，C2C",
    "bench_drift_c2c": "基准组合 buy-and-hold 收益 [2,61]，C2C",
    "bench_ann_o2o": "基准组合 buy-and-hold 收益 [0,1]，O2O",
    "bench_drift_o2o": "基准组合 buy-and-hold 收益 [2,61]，O2O",
    "port25": "所属 size×B/M (Book-to-Market) 基准组合编号 1-25 = (size_q−1)×5 + bm_q",
    "n_days_ann_c2c": "ANN 窗口内实际有 C2C 收益记录的天数（满窗=2）",
    "n_days_drift_c2c": "DRIFT 窗口内实际有 C2C 收益记录的天数（满窗=60）",
    "n_days_ann_o2o": "同上，O2O 口径", "n_days_drift_o2o": "同上，O2O 口径",
    # ---- 核心自变量 ----
    "sue": "★ SUE (Standardized Unexpected Earnings) 标准化盈余意外，HLT 2009 记作 FE (Forecast Error)：(实际EPS − 分析师共识预测) / 拆股调整后财季末股价",
    "sue_dec": "★ SUE decile 十分位（按公告日所在日历季度独立分组，1=最负意外，10=最正）",
    "consensus_f": "Consensus Forecast 分析师共识预测 F（公告前 60 天内每位分析师最新一条的中位数）",
    "price_adj": "SUE 分母 P：财季末股价，已折到与 EPS 相同的拆股基准 = |mthprc| / mthcumfacpr (Cumulative Factor to Adjust Price)",
    "prc_unadj": "财季末股价原值（HLT 的 <$1 清洗规则用的是这个）",
    "n_analyst_sue": "Number of Analysts in consensus：算这次共识用到的分析师人数（诊断用，非控制变量）",
    "fcst_last_dt": "共识中最新一条预测的日期",
    # ---- 控制变量 ----
    "size_dec": "◆ SIZE (Firm Size decile) 规模十分位：年初（上年12月末）市值，用 NYSE 股票算断点",
    "me_jan": "ME (Market Equity) 年初市值原值，千美元",
    "bm_dec": "◆ BM (Book-to-Market) decile 账面市值比十分位，用 NYSE 股票算断点",
    "bm_raw": "BM (Book-to-Market) 连续值 = BE×1000 / 上年12月末 ME（单位对齐：百万→千）",
    "be": "BE (Book Equity) 账面权益 = SEQ (Stockholders Equity) + TXDITC (Deferred Taxes) − PS (Preferred Stock)，百万美元",
    "lnanalyst": "◆ LNANALYST (Log of 1 + Number of Analysts) = log(1 + 公告前 365 天内出过预测的不同分析师数)",
    "n_analyst_cover": "上述分析师数的原始计数",
    "lag": "◆ LAG (Reporting Lag) 报告滞后 = anndats − pends，天",
    "lag2": "◆ LAG² 报告滞后的平方", "lag3": "◆ LAG³ 报告滞后的三次方",
    "io": "◆ IO (Institutional Ownership) 机构持股比例 = 13F Σshares / (shrout×1000)",
    "evol": "◆ EVOL (Earnings Volatility) 盈余波动率：过去 16 财季(4年)内 Δ₄EPS 的样本标准差，美元/股",
    "epersist": "◆ EPERSIST (Earnings Persistence) 盈余持续性：同一 Δ₄EPS 序列的一阶自相关，−1~1",
    "turn": "◆ TURN (Share Turnover) 换手率：过去 12 个月的月均 mthvol/(shrout×1000)",
    # ---- 标记与诊断 ----
    "flag_lag_bad": "LAG < 0 或 > 180 天（异常报告滞后）",
    "flag_same_day_multi": "同一 permno 同一天公告多个财季（补报）",
    "is_latest_pends_on_day": "该公告日中是否为当期财季（补报里最新的一期）",
    "flag_link_extended": "IBES 链接在连接表末期仍有效（绝大多数本就能匹配，无风险）",
    "flag_beyond_link_end": "公告日晚于 2025-12-31，只因延长链接才存在（有错配风险）",
    "flag_short_window": "ANN 不足 2 天 或 DRIFT 不足 60 天",
    "flag_sue_dropped": "被 SUE 清洗规则剔除（股价<$1 或 |e|/|F| 超过股价）",
    "ann_on_nontrading": "公告落在非交易日（已顺延到下一交易日）",
    "td0_gap_days": "td0 − anndats 的天数（正常为 0-3；早于日历起点的事件会很大）",
    "flag_pre_calendar": "公告日早于交易日历起点 1996-01-02（CAR 无意义，已置空）",
    "flag_td0_gap": "td0 与公告日相隔 >7 天或早于日历起点 → CAR 已置空（5,628 个事件）",
    "io_stale_days": "IO 数据距公告日的天数（>200 天已置为缺失）",
    "eps_stale_days": "EVOL/EPERSIST 数据距公告日的天数（>400 天已置为缺失）",
    "rdate": "IO 所用的 13F 报告期",
    "datadate": "EVOL/EPERSIST 所用的 Compustat 财季末",
}

panel = pd.read_parquet(f"{BUILD}/pead_panel.parquet")
rows = []
for c in panel.columns:
    s = panel[c]
    med = ""
    if s.dtype.kind in "fiu" or str(s.dtype).startswith(("Int", "Float")):
        try:
            med = f"{pd.to_numeric(s, errors='coerce').median():.4g}"
        except Exception:
            med = ""
    rows.append({"列名": c, "类型": str(s.dtype), "覆盖率": f"{s.notna().mean():.1%}",
                 "中位数": med, "说明": DICT.get(c, "⚠️ 未登记")})
dd = pd.DataFrame(rows)
dd.to_csv(f"{BUILD}/data_dictionary.csv", index=False)

missing = dd.loc[dd["说明"].str.startswith("⚠️"), "列名"].tolist()
print(f"pead_panel: {len(panel):,} 行 × {panel.shape[1]} 列 | 字典已覆盖 {panel.shape[1]-len(missing)} 列")
if missing:
    print("未登记的列:", missing)
print(f"\n→ {BUILD}/data_dictionary.csv\n")
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 45):
    print(dd.to_string(index=False))

pead_panel: 517,955 行 × 72 列 | 字典已覆盖 72 列

→ build/data_dictionary.csv

                    列名             类型    覆盖率        中位数                                                                                                              说明
                   eid          int64 100.0%   2.59e+05                                                                           事件唯一编号（= events.parquet 的行号，用于回溯各中间表）
                permno          int64 100.0%  7.784e+04                                                              PERMNO (CRSP Permanent Number) CRSP 股票永久标识 —— 本表主键
                ticker         string 100.0%                                                                    I/B/E/S Ticker：I/B/E/S 内部永久标识（≤6 位字母），不是交易所代码（那是 oftic）
               anndats datetime64[ns] 100.0%                                                                                            Announcement Date 盈余公告日 —— 事件锚点
                   td0 datetime64[ns] 100.0%                                            

### 8.1 完整数据字典（表格视图）

上面是文本版（说明列有截断）。下面把 `build/data_dictionary.csv` 完整渲染成表格 ——
72 行全部展开、说明列不截断，方便查阅与复制。

In [22]:
dd_view = pd.read_csv(f"{BUILD}/data_dictionary.csv")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", None):
    display(dd_view.style
            .hide(axis="index")
            .set_properties(**{"text-align": "left", "font-size": "12px"})
            .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

列名,类型,覆盖率,中位数,说明
eid,int64,100.0%,259000.000000,事件唯一编号（= events.parquet 的行号，用于回溯各中间表）
permno,int64,100.0%,77840.000000,PERMNO (CRSP Permanent Number) CRSP 股票永久标识 —— 本表主键
ticker,string,100.0%,nan,I/B/E/S Ticker：I/B/E/S 内部永久标识（≤6 位字母），不是交易所代码（那是 oftic）
anndats,datetime64[ns],100.0%,nan,Announcement Date 盈余公告日 —— 事件锚点
td0,datetime64[ns],100.0%,nan,事件日 = 公告日当天或之后的第一个交易日
td0_idx,int64,100.0%,3063.000000,td0 在交易日历中的下标（CAR 窗口用它取值）
pends,datetime64[ns],100.0%,nan,Period End Date 财季结束日
actual_eps,Float64,93.9%,0.160000,实际 EPS (Earnings Per Share)，I/B/E/S，已折算到最新拆股基准
anntims,string,100.0%,nan,Announcement Time 公告时间（可判断盘前/盘后）
lag,int64,100.0%,33.000000,◆ LAG (Reporting Lag) 报告滞后 = anndats − pends，天
